# Module 15: Diagnostics and Transformations

Diagnostics ask where a model is systematically wrong. We use residual plots,
a heteroskedasticity test, nonlinear terms, robust uncertainty, and influence
measures. The goal is a justified revision—not a ritual sequence of tests.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import root_mean_squared_error
from statsmodels.stats.diagnostic import het_breuschpagan

sns.set_theme(style="whitegrid")

In [ ]:
hotels = pd.read_csv("data/raw/hotels_course.csv")
vienna = hotels.query("city == 'Vienna' and accommodation == 'Hotel'").copy()
linear = smf.ols("log_price ~ distance_miles + rating + C(stars)", data=vienna).fit()

## 1. Look for structure left in the residuals

In [ ]:
diagnostics = vienna.assign(fitted=linear.fittedvalues, residual=linear.resid)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=diagnostics, x="fitted", y="residual", alpha=0.65, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
sm.qqplot(diagnostics["residual"], line="45", fit=True, ax=axes[1])
axes[0].set(title="Residuals versus fitted values")
axes[1].set(title="Normal Q-Q plot")
plt.tight_layout()
plt.show()

In [ ]:
bp_stat, bp_pvalue, _, _ = het_breuschpagan(linear.resid, linear.model.exog)
pd.Series({"Breusch-Pagan statistic": bp_stat, "p-value": bp_pvalue}).round(4)

A small p-value is evidence that residual variance changes with the predictors.
Robust standard errors adjust uncertainty estimates; they do not fix a poor
conditional-mean specification.

In [ ]:
conventional = linear.bse.rename("conventional_se")
robust = linear.get_robustcov_results(cov_type="HC3")
robust_se = pd.Series(robust.bse, index=linear.params.index, name="robust_se")
pd.concat([conventional, robust_se], axis=1).round(4)

## 2. Test a substantively plausible nonlinear relationship

In [ ]:
quadratic = smf.ols(
    "log_price ~ distance_miles + I(distance_miles ** 2) + rating + C(stars)",
    data=vienna,
).fit(cov_type="HC3")

In [ ]:
pd.DataFrame(
    {
        "model": ["linear distance", "quadratic distance"],
        "aic": [linear.aic, quadratic.aic],
        "rmse_log": [
            root_mean_squared_error(vienna["log_price"], linear.fittedvalues),
            root_mean_squared_error(vienna["log_price"], quadratic.fittedvalues),
        ],
    }
).round(3)

In [ ]:
distance_grid = pd.DataFrame(
    {
        "distance_miles": np.linspace(0.2, 8, 100),
        "rating": 4.0,
        "stars": 4.0,
    }
)
distance_grid["linear"] = np.exp(linear.predict(distance_grid))
distance_grid["quadratic"] = np.exp(quadratic.predict(distance_grid))

In [ ]:
plot_grid = distance_grid.melt("distance_miles", var_name="model", value_name="predicted_price")
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=plot_grid, x="distance_miles", y="predicted_price", hue="model", ax=ax)
ax.set(title="Functional form changes the implied relationship", ylabel="Predicted price")
plt.show()

## 3. Back-transform carefully and inspect influence

In [ ]:
smearing = np.mean(np.exp(quadratic.resid))
vienna["predicted_price"] = np.exp(quadratic.fittedvalues) * smearing
vienna["price_residual"] = vienna["price"] - vienna["predicted_price"]
vienna[["price", "predicted_price", "price_residual"]].describe().round(1)

In [ ]:
influence = quadratic.get_influence()
vienna["cooks_distance"] = influence.cooks_distance[0]
influential = vienna.nlargest(8, "cooks_distance")[
    ["hotel_id", "price", "distance_miles", "rating", "cooks_distance"]
]
influential.round(3)

Influential observations deserve investigation, not automatic deletion. Check
whether they are errors, unusual but valid cases, or evidence that the model
misses a relevant feature.

## Session 2 reconstruction handoff

Open `prompts/session02-specification-diagnostics-reconstruction.md`. The audit
will focus on reference categories, interactions, transformations, residual
evidence, and whether the generated interpretation overclaims causality.

### Optional extension

Replace the quadratic with `bs(distance_miles, df=4)` from Patsy and compare
the fitted curve. Greater flexibility must be justified and validated.

In [ ]:
spline = smf.ols(
    "log_price ~ bs(distance_miles, df=4) + rating + C(stars)", data=vienna
).fit(cov_type="HC3")
pd.Series({"quadratic_aic": quadratic.aic, "spline_aic": spline.aic}).round(2)